## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 12 · Inference walkthrough and project handoff

This notebook loads the selected artifact and scores an already-engineered row. It is an offline walkthrough, not a real-time service: behavioral features need historical state and must use only events available before a transaction.

In [2]:
import joblib
import numpy as np
import pandas as pd

artifact_path=MODEL_DIR/"best_experiment_model.joblib"; data_path=INTERMEDIATE_DIR/"test_features.pkl"
if not artifact_path.exists() or not data_path.exists(): raise FileNotFoundError("Run notebooks 04 and 06 first.")
artifact=joblib.load(artifact_path); test_features=pd.read_pickle(data_path); features=list(artifact["feature_cols"])
missing=sorted(set(features)-set(test_features.columns))
if missing: raise ValueError(f"Test data lacks required model inputs: {missing[:10]}")
print(f"Loaded {type(artifact['model']).__name__} with {len(features)} features.")

Loaded XGBClassifier with 420 features.


## 1. Score one competition example

In [3]:
row=test_features.iloc[[0]]; X=row[features].replace([np.inf,-np.inf],np.nan)
if artifact.get("imputer") is not None: X=artifact["imputer"].transform(X)
if artifact.get("scaler") is not None: X=artifact["scaler"].transform(X)
p=float(artifact["model"].predict_proba(X)[0,1])
{"transaction_id":row.TransactionID.iloc[0] if "TransactionID" in row else None,"model_probability_output":p,"risk_score_0_to_100":round(100*p,2),"model_name":artifact.get("experiment_name",type(artifact["model"]).__name__)}

{'transaction_id': np.int64(3663549),
 'model_probability_output': 0.032183337956666946,
 'risk_score_0_to_100': 3.22,
 'model_name': 'XGBoost'}

## 2. Safe inference contract

A real service should validate required inputs, reject or explicitly handle invalid values, record a request id and model version, and return documented decision information. Malformed input must not silently become a low-risk score. The precomputed row is only a demonstration of the expected model matrix.

In [4]:
numeric=row[features].select_dtypes(include=[np.number])
print("Required model features:",len(features))
print("Missing required columns:",len(set(features)-set(row.columns)))
print("Non-finite numeric values:",int((~np.isfinite(numeric.to_numpy())).sum()))

Required model features: 420
Missing required columns: 0
Non-finite numeric values: 192


## 3. Notebook path and handoff

- **01–03:** data ingestion, validation, exploration.
- **04:** time-aware feature engineering.
- **05–06:** baseline and model experiments.
- **07–08:** evaluation and risk scoring.
- **09–10:** explanations and decision policy.
- **11–12:** monitoring and an offline inference walkthrough.

The production handoff still requires a point-in-time feature pipeline, a later untouched temporal test set, a service API, access controls, and operational review. These notebooks do not claim to provide a live fraud system.